In [ ]:
from pathlib import Path


import torch
import pandas as pd
from tqdm.notebook import tqdm

from rtnls_inference import (
    RegressionEnsemble,
)
from rtnls_fundusprep.mask_extraction import CFIBounds as Bounds

In [ ]:
ds_path = Path("../samples/fundus")

# input folders. these are the folders where we stored the preprocessed images
rgb_path = ds_path / "rgb"
device = torch.device("cuda:0")  # device to use for inference

In [ ]:
rgb_paths = sorted(list(rgb_path.glob("*.png")))

In [ ]:
rgb_paths

In [ ]:
ensemble = RegressionEnsemble.from_release("odfd_march25.pt").to(device)

dataloader = ensemble._make_inference_dataloader(
    rgb_paths,
    num_workers=8,
    preprocess=True,
    batch_size=8,
)

In [ ]:
from rtnls_inference.utils import decollate_batch


output_ids, outputs = [], []
with torch.no_grad():
    for batch in tqdm(dataloader):
        if len(batch) == 0:
            continue

        im = batch["image"].to(device)
        val = ensemble.forward(im).mean(dim=0).detach().cpu() # average the model dimension
      
        batch['val'] = val
        items = decollate_batch(batch)

        for item in items:
            scaling_factor = item['metadata']['bounds']['radius'] / 512
            outputs.append(item['val'] * 1024 * scaling_factor)
            output_ids.append(item['id'])


In [ ]:
df = pd.DataFrame(outputs, index=output_ids)

In [ ]:
df